# LightGBM Classifier — Step-by-Step Guide

LightGBM (Light Gradient Boosting Machine) builds many decision trees sequentially — each tree learns from the mistakes of the previous one. It is fast, handles class imbalance well, and typically outperforms logistic regression on tabular data.

**Baseline to beat:** Logistic Regression ROC-AUC = 0.8574

---

## Step 1 — Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
from lightgbm import LGBMClassifier

print('✅ Imports done')

---
## Step 2 — Load & Split

In [ ]:
df = pd.read_csv('data/features_encoded_train.csv')

X = df.drop(columns=['target_bank_account'])
y = df['target_bank_account']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training rows  : {X_train.shape[0]:,}')
print(f'Validation rows: {X_val.shape[0]:,}')

# Note: no scaling needed — LightGBM is tree-based and scale-invariant

---
## Step 3 — Train LightGBM

Key parameters:
- `class_weight='balanced'` — compensates for the 86/14 class imbalance
- `n_estimators` — number of trees to build (more = more powerful but slower)
- `learning_rate` — how much each tree corrects the previous one (lower = more careful)
- `max_depth` — maximum depth of each individual tree

In [ ]:
model = LGBMClassifier(
    class_weight='balanced',
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    verbose=-1          # suppress LightGBM training logs
)

model.fit(X_train, y_train)

print('✅ Model trained')

---
## Step 4 — Evaluate

In [ ]:
y_proba = model.predict_proba(X_val)[:, 1]
y_pred  = model.predict(X_val)

auc = roc_auc_score(y_val, y_proba)
print(f'ROC-AUC : {auc:.4f}  (baseline logistic regression: 0.8574)')
print(f'Improvement: {"+" if auc > 0.8574 else ""}{auc - 0.8574:.4f}')
print()
print(classification_report(y_val, y_pred,
      target_names=['No bank account', 'Has bank account']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y_val, y_proba)
axes[0].plot(fpr, tpr, color='#854F0B', lw=2, label=f'LightGBM (AUC = {auc:.3f})')
axes[0].plot([0,1],[0,1],'k--',lw=1, label='Baseline logistic regression (0.857)')
axes[0].fill_between(fpr, tpr, alpha=0.08, color='#854F0B')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=9)
axes[0].spines[['top','right']].set_visible(False)

# Confusion Matrix
cm = confusion_matrix(y_val, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['No bank account', 'Has bank account']).plot(
    ax=axes[1], colorbar=False, cmap='Oranges')
axes[1].set_title('Confusion Matrix (validation set)', fontweight='bold')

plt.suptitle('LightGBM Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 5 — Feature Importance

LightGBM ranks features by how much they reduce prediction error across all trees — similar to the decision tree importance chart, but aggregated over 300 trees instead of one.

In [ ]:
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

importance_df['label'] = (importance_df['feature']
    .str.replace('ohe_job_', 'job: ', regex=False)
    .str.replace('ohe_country_', 'country: ', regex=False)
    .str.replace('bin_location_type', 'Urban location', regex=False)
    .str.replace('bin_cellphone_access', 'Cellphone access', regex=False)
    .str.replace('bin_gender', 'Gender (Male=1)', regex=False)
    .str.replace('ord_education_level', 'Education level', regex=False)
    .str.replace('age_of_respondent', 'Age', regex=False)
)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(importance_df['label'], importance_df['importance'],
        color='#854F0B', edgecolor='white', height=0.6)
ax.set_title('Feature importance — LightGBM', fontweight='bold', pad=12)
ax.set_xlabel('Importance score')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## Step 6 — Summary

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print('=' * 52)
print('  LIGHTGBM MODEL SUMMARY')
print('=' * 52)
print(f'  ROC-AUC    : {auc:.4f}  (baseline: 0.8574)')
print(f'  Accuracy   : {accuracy_score(y_val, y_pred):.4f}')
print(f'  Precision  : {precision_score(y_val, y_pred):.4f}')
print(f'  Recall     : {recall_score(y_val, y_pred):.4f}')
print(f'  F1         : {f1_score(y_val, y_pred):.4f}')
print('=' * 52)
print()
print('Key parameters used:')
print('  n_estimators  = 300   (number of trees)')
print('  learning_rate = 0.05  (step size per tree)')
print('  max_depth     = 6     (depth of each tree)')
print()
print('Next steps to try:')
print('  → Tune n_estimators, learning_rate, max_depth with GridSearchCV')
print('  → Lower the decision threshold (as done in the baseline notebook)')
print('  → Try num_leaves parameter — LightGBM grows leaf-wise, not depth-wise')